# Study B: information gaps, alternatives, and finite feature families

Study B combines population examples, fixed-alternative interval coverage, local power, descriptive weight stress, and a prespecified finite family of tests. These components answer different questions and should be interpreted separately.

Smoke mode runs seventy main replicates, thirty descriptive profiles, and five finite-family replicates. Paper mode runs eight thousand four hundred main replicates, six hundred profiles, and four hundred finite-family replicates.

The notebook calls the shared workflow and reads its saved results. It does not implement a second fitting routine. Generated outputs go to `results/smoke` or `results/paper`; the frozen `reference_results` directory is not overwritten.

Run all cells from top to bottom after changing the mode. Smoke mode checks execution and output structure. Its tiny simulation samples are not evidence for the manuscript's statistical conclusions.


In [1]:
# Change MODE to "paper" to run the complete manuscript experiment.
# Publication figures require LaTeX and the configured image-conversion tools.
MODE = "smoke"
FIGURES = False
assert MODE in {"smoke", "paper"}


In [2]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
REPO = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "workflow.py").is_file() and (p / "code" / "gid_pipeline.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("Open this notebook from the repository root or its notebooks directory.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from workflow import run_study


def read_json(path):
    return json.loads(Path(path).read_text())


def require_finite(frame, columns):
    values = frame.loc[:, columns].apply(pd.to_numeric, errors="raise").to_numpy()
    assert np.isfinite(values).all(), f"Nonfinite values in {columns}"


def show_run_figure(relative_path):
    if not FIGURES:
        print("Figure generation is off. Set FIGURES=True and rerun to create a figure from this run.")
        return
    figure = RUN / relative_path
    if not figure.is_file():
        raise FileNotFoundError(f"The requested current-run figure was not generated: {figure}")
    display(Markdown(f"Figure from `{figure.relative_to(REPO)}` ({MODE} mode)."))
    display(Image(filename=str(figure)))

print(f"Repository root located: {REPO.name}")
print(f"Mode: {MODE}; figures: {FIGURES}")


Repository root located: github
Mode: smoke; figures: False


In [3]:
RUN = Path(run_study("b", mode=MODE, figures=FIGURES)).resolve()
assert RUN == (REPO / "results" / MODE).resolve()
assert RUN != (REPO / "reference_results").resolve()
print(f"Reading generated results from {RUN.relative_to(REPO)}")


smoke: study_ab-b


smoke: study_b_supplement


Reading generated results from results/smoke


## Population geometry and the fourth-harmonic example

Raw second-moment anisotropy can arise under a correctly specified von Mises-Fisher model. The residual second moment subtracts the structure already implied by the fitted first moment.

The circular fourth-harmonic example illustrates another limit. Several consecutive zero population gaps do not imply that all directional structure has been described. A richer feature family can still add information.


In [4]:
geometry = pd.read_csv(RUN / "study_b" / "sphere_population_geometry.csv")
hierarchy = pd.read_csv(RUN / "study_b" / "cos4_population_hierarchy.csv")
assert len(geometry) == 4 and len(hierarchy) == 4
require_finite(geometry, ["I1", "I2", "raw_anisotropy", "residual_anisotropy"])
require_finite(hierarchy, ["deficit", "gap", "moment_error"])
display(geometry[["law", "resultant", "raw_anisotropy", "residual_anisotropy", "I1", "I2", "I2_refinement"]])
display(hierarchy)


,law,resultant,raw_anisotropy,residual_anisotropy,I1,I2,I2_refinement
0,uniform,2.634263e-15,1.249100e-14,5.613802e-17,0.000000,0.000000,0.000000e+00
1,vmf8,8.750002e-01,5.485836e-01,2.414999e-13,1.772591,0.000000,0.000000e+00
2,antipodal8,4.302276e-15,5.485836e-01,5.485836e-01,0.000000,1.065902,1.687539e-14
3,girdle12,1.044120e-15,3.572185e-01,3.572185e-01,0.000000,0.863249,5.595524e-14


,level,deficit,gap,moment_error
0,1,0.000000,0.000000,5.204170e-17
1,2,0.000000,0.000000,4.163336e-17
2,3,0.000000,0.000000,6.196215e-17
3,4,0.167043,0.167043,4.363176e-14


## Fixed alternatives and local power

The delta-method interval result concerns nonzero population gaps. Near-zero or local-alternative coverage is diagnostic and does not have the same justification. Local equivalence also does not guarantee equal finite-sample power for gap, score, and Wald tests.


In [5]:
summary = pd.read_csv(RUN / "study_b" / "power_estimation_summary.csv")
manifest = read_json(RUN / "study_b" / "numerical_manifest.json")
main_files = sorted((RUN / "study_b").glob("fixed_*_replicates.csv")) + sorted((RUN / "study_b").glob("local_*_replicates.csv"))
replicates = pd.concat([pd.read_csv(p) for p in main_files], ignore_index=True)
expected_main = 70 if MODE == "smoke" else 8400
assert len(summary) == 14 == len(main_files)
assert len(replicates) == expected_main == manifest["total_replicates"]
assert summary["repeats"].sum() == expected_main
assert not replicates.duplicated(["cell", "replicate"]).any()
display(replicates.groupby(["status", "refinement_status"], dropna=False).size().rename("replicates").to_frame())
assert replicates["valid"].astype(str).str.lower().eq("true").all(), "Inspect the saved invalid fits."
require_finite(replicates, ["gap", "se", "lower", "upper", "p_gap", "moment_error"])
assert replicates["p_gap"].between(0, 1).all()
assert (replicates["lower"] <= replicates["upper"]).all()

display(summary.loc[summary["regime"].eq("fixed"),
    ["b", "n", "repeats", "population_gap", "bias", "rmse", "coverage", "coverage_lo", "coverage_hi"]])
display(summary.loc[summary["regime"].eq("local"),
    ["b", "n", "repeats", "p_gap_rejection", "p_score_rejection", "p_wald_rejection", "asymptotic_local_power"]])
show_run_figure("study_b/power_coverage.png")


,,replicates
status,refinement_status,
ok,ok,70


,b,n,repeats,population_gap,bias,rmse,coverage,coverage_lo,coverage_hi
0,0.3,200,5,0.008448,0.000309,0.006034,1.0,0.565518,1.000000
1,0.6,200,5,0.029424,0.015324,0.030223,0.8,0.375535,0.963776
2,0.3,800,5,0.008448,0.005138,0.006109,1.0,0.565518,1.000000
3,0.6,800,5,0.029424,-0.001415,0.011436,0.8,0.375535,0.963776


,b,n,repeats,p_gap_rejection,p_score_rejection,p_wald_rejection,asymptotic_local_power
4,0.0,200,5,0.2,0.2,0.2,0.050000
5,1.0,200,5,0.0,0.0,0.0,0.066292
6,2.0,200,5,0.0,0.0,0.0,0.119338
7,3.0,200,5,0.0,0.0,0.0,0.216897
8,4.0,200,5,0.2,0.2,0.2,0.359021
9,0.0,800,5,0.0,0.0,0.0,0.050000
10,1.0,800,5,0.0,0.0,0.0,0.066292
11,2.0,800,5,0.2,0.2,0.2,0.119338
12,3.0,800,5,0.2,0.2,0.2,0.216897
13,4.0,800,5,0.0,0.0,0.0,0.359021


Figure generation is off. Set FIGURES=True and rerun to create a figure from this run.


## Descriptive weight stress

The Pareto weights in this experiment have infinite second moment. The saved profiles illustrate changes in fitted information and weight concentration. They do not support Gaussian covariance calibration or population p-values.

Each sample is analyzed with equal and Pareto weights. The profile count includes both analyses of the same sample.


In [6]:
stress = pd.read_csv(RUN / "study_b" / "pareto_stress_replicates.csv")
stress_summary = pd.read_csv(RUN / "study_b" / "pareto_stress_summary.csv")
expected_profiles = 30 if MODE == "smoke" else 600
assert len(stress) == expected_profiles
assert len(stress_summary) == 6
assert stress_summary["repeats"].sum() == expected_profiles
assert stress["success"].astype(str).str.lower().eq("true").all()
require_finite(stress, ["ess", "max_weight", "I1", "I2", "I3", "I4", "D4", "moment_error"])
display(stress_summary[["law", "weights", "repeats", "ess_mean", "max_weight_mean", "I1_mean", "I2_mean", "I3_mean", "I4_mean"]])


,law,weights,repeats,ess_mean,max_weight_mean,I1_mean,I2_mean,I3_mean,I4_mean
0,unimodal,equal,5,400.000000,0.002500,1.052952,0.002715,0.003818,0.002548
1,unimodal,pareto,5,54.072206,0.138347,1.081197,0.024341,0.021553,0.011223
2,antipodal,equal,5,400.000000,0.002500,0.000818,0.661869,0.001116,0.010579
3,antipodal,pareto,5,98.029585,0.072666,0.006109,0.676195,0.005325,0.021620
4,trimodal,equal,5,400.000000,0.002500,0.002338,0.003609,0.161186,0.003688
5,trimodal,pareto,5,86.480136,0.091513,0.015342,0.013553,0.194842,0.017961


## A fixed family of adjacent-gap and tail tests

The family contains the first four adjacent gaps and the tail from the first to fourth fitted level. Holm adjustment is applied across all five tests. Stopping after the first nonsignificant gap would be a different procedure.

The first-gap nonrejection count below uses its **unadjusted** p-value. The family-wise count uses Holm-adjusted p-values for the first three gaps, which are true nulls in this example. Finite-sample exact error control is not claimed.


In [7]:
tail = pd.read_csv(RUN / "study_b" / "finite_tail_holm_replicates.csv")
tail_summary = read_json(RUN / "study_b" / "finite_tail_holm_summary.json")
expected_tail = 5 if MODE == "smoke" else 400
assert len(tail) == expected_tail == tail_summary["repeats"]
validity = ["valid1", "valid2", "valid3", "valid4", "valid_tail"]
assert all(tail[c].astype(str).str.lower().eq("true").all() for c in validity)
p_columns = ["p1", "p2", "p3", "p4", "p_tail"]
holm_columns = ["holm_I1", "holm_I2", "holm_I3", "holm_I4", "holm_tail"]
require_finite(tail, p_columns + holm_columns)
assert tail[p_columns + holm_columns].ge(0).all().all()
assert tail[p_columns + holm_columns].le(1).all().all()
assert int((tail["p1"] > 0.05).sum()) == tail_summary["first_gap_nonrejections"]
family_errors = int(tail[["holm_I1", "holm_I2", "holm_I3"]].lt(0.05).any(axis=1).sum())
assert family_errors == tail_summary["holm_any_true_null_rejections"]
display(pd.DataFrame({k: tail_summary[k] for k in holm_columns}).T)
display(pd.Series({
    "replicates": expected_tail,
    "unadjusted first-gap nonrejections": tail_summary["first_gap_nonrejections"],
    "Holm rejection of any true null": family_errors,
    "failed replicate analyses": tail_summary["failures"],
}, name="count").to_frame())


,rejections,rate,wilson_low,wilson_high
holm_I1,1.0,0.2,0.036224,0.624465
holm_I2,0.0,0.0,0.000000,0.434482
holm_I3,0.0,0.0,0.000000,0.434482
holm_I4,5.0,1.0,0.565518,1.000000
holm_tail,5.0,1.0,0.565518,1.000000


,count
replicates,5
unadjusted first-gap nonrejections,4
Holm rejection of any true null,1
failed replicate analyses,0
